# Voz en off en los Reels, con el dibujo en bucle

Los Reels son **canal propio**: su identidad visual no se toca. Lo único que se trae de la serie
larga es el narrador, que es justo lo que hace que un espectador reconozca de quién es el vídeo
aunque el formato cambie.

El problema que resuelve este notebook es el del ritmo. En `reel_common.Timeline` cada escena dura
un número fijo de fotogramas y la animación tiene que caber dentro, así que todo pasa muy rápido:
18 segundos para seis escenas. Aquí se invierte el mando: **la escena dura lo que tarde su frase**,
y el dibujo se repite debajo tantas veces como haga falta. La curva de error se pinta, termina,
vuelve a empezar, y mientras la voz explica tranquilamente qué significa.

El motor nuevo está en `reel_voz.py`. Las funciones de dibujo **no cambian**: una escena ya escrita
sirve tal cual, porque recibe el fotograma del ciclo en curso en vez del fotograma absoluto.

## Las dos identidades, para no mezclarlas

| | Serie larga | Reels |
|---|---|---|
| Formato | 16:9, 4–14 min | 1080×1920, 18–23 s |
| Fondo | `#111318` | `#080C12` (más frío) |
| Titulares | Inter, 48 px, minúsculas | Archivo Black, 86 px, **mayúsculas** |
| Cuerpo | Inter 32 px | Inter 46 px |
| Lógica del color | **semántica**: cada concepto tiene el suyo fijo (datos azul, gradiente naranja, pesos violeta, error rojo) | **jerárquica**: dorado `#F4BE4E` = la idea clave, azul `#6EA8FF` = los datos, verde/rojo = acierto/fallo |
| Texto en pantalla | máx. 7 palabras | máx. 2 líneas de ~6 palabras |
| Zona útil | todo el cuadro | y entre 260 y 1480 (Instagram tapa el resto) |
| Audio | voz + música por tramos + QA con Whisper | sintetizado con NumPy, **sin voz** |

Lo que de verdad las separa no son los colores, es la lógica: en la serie el color **significa**
algo y se respeta durante doce minutos; en un Reel de veinte segundos el color **ordena** la
atención. Por eso no conviene unificar las paletas, sino una sola cosa: que el rojo no sea error
en un sitio y acierto en el otro.

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

RAIZ = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "pipeline" / "produce.py").exists()), None)
if RAIZ is None:
    raise RuntimeError("Abre el notebook desde dentro de series-redes-neuronales/")
if Path(sys.prefix).resolve() != (RAIZ / ".venv").resolve():
    raise RuntimeError(f"Kernel equivocado: {sys.executable}. "
                       f"Selecciona 'series-redes-neuronales (.venv 3.13)' o {RAIZ / '.venv/bin/python'}")

# reel_common busca fonts/, work/ y out/ en el directorio actual, así que se trabaja desde dentro
# del proyecto de los Reels. Es su convención; no la cambiamos desde aquí.
REELS = RAIZ / "redes_neuronales_vscode"
os.chdir(REELS)
sys.path.insert(0, str(REELS))
sys.path.insert(0, str(RAIZ))   # para reusar el modelo de ritmo del narrador

import numpy as np
import yaml
from PIL import Image, ImageDraw
from IPython.display import Audio as Reproductor, Video, display

import reel_common as rc
import reel_voz as rv

voz = yaml.safe_load((RAIZ / "config/voz.yaml").read_text(encoding="utf-8"))
print(f"{rc.W}x{rc.H} a {rc.FPS} fps · zona útil y∈[{rc.SAFE_TOP}, {rc.SAFE_BOTTOM}]")
print(f"narrador: {voz['motor']} · {voz['qwen']['modelo']} · velocidad {voz['velocidad']}")

## 1. El guion

Una idea por escena, y el ciclo visual es lo que el ojo sigue mientras la voz explica. La regla
práctica: **el ciclo corto (2–3 s) y la frase larga**. Si el ciclo dura lo mismo que la frase,
vuelves a tener el problema de siempre.

In [ ]:
GUION = [
 {"id": "curva", "ciclo_s": 2.2, "texto":
  "Mira caer el error de una red mientras aprende. Falla en casi todo, "
  "y cada paso lo corrige un poco, hasta que se queda plana."},

 {"id": "examen", "ciclo_s": 2.6, "texto":
  "Pero hay dos curvas. La de abajo mide lo que ya ha visto; la de arriba, "
  "lo que no. Cuando esa deja de bajar, está memorizando."},

 {"id": "cierre", "ciclo_s": 1.5, "texto":
  "La próxima vez que veas una curva, mira la de arriba."},
]

# El modelo de ritmo del narrador (voz_qa.duracion_modelo) predice la duración sin gastar GPU:
# duración ≈ a + b·caracteres, medido sobre las frases aprobadas del episodio 00.
from lib.texto import numeros_a_palabras

qa = yaml.safe_load((RAIZ / "config/pipeline.yaml").read_text(encoding="utf-8"))["voz_qa"]
a, b = qa["duracion_modelo"]
previsto = 0.0
for e in GUION:
    n = len(numeros_a_palabras(e["texto"]))
    d = (a + b * n) / voz["velocidad"]
    previsto += d + 0.35
    print(f"  {e['id']:8} {n:4} caracteres → {d:4.1f} s · ciclo {e['ciclo_s']} s → {d / e['ciclo_s']:.1f} vueltas")
print(f"\ntotal previsto: {previsto:.1f} s · el formato pide 18-23 s")
if previsto > 23:
    print("[ojo] recorta texto antes de generar la voz: un Reel largo se abandona a la mitad")


## 2. La voz

La genera Qwen3-TTS con tu voz de referencia, en `.venv-qwen`, y se estira a la `velocidad` de
`config/voz.yaml` (0,92 ahora mismo) **antes** de medirla: lo que mide la línea de tiempo es el
audio que se va a publicar.

Necesita la GPU libre. Las frases ya generadas con el mismo texto y la misma velocidad no se
repiten, así que puedes volver a ejecutar esta celda sin coste.

In [ ]:
escenas = rv.narrar(GUION, RAIZ, salida=Path("work/voz_demo"))
for e in escenas:
    print(f"  {e['id']:8} {e['duracion']:5.2f} s · {e['texto'][:60]}…")
print(f"\ntotal de voz: {sum(e['duracion'] for e in escenas):.1f} s")
for e in escenas:
    display(f"── {e['id']}")
    display(Reproductor(str(e["wav"])))

## 3. Las escenas

La curva se pinta avanzando `upto` sobre el eje logarítmico, de la iteración 1 a la 3.000, en los
~2 segundos del ciclo. Al terminar vuelve a 1 y `reel_voz` funde el salto para que no sea un corte.

Los datos son sintéticos a propósito: esto prueba el ritmo y la voz, no un entrenamiento. Cuando
lo lleves a un Reel real, `xs`/`ys` salen del `.pkl` de su `train_*.py`.

In [ ]:
XMAX = 3000
XS = np.unique(np.round(np.logspace(0, np.log10(XMAX), 200)).astype(int))
_rng = np.random.default_rng(7)
_T = np.log10(XS) / np.log10(XMAX)        # 0 a 1 recorriendo el eje, que es logarítmico

def _curva(alto, suelo, ruido, codo=1.6, subida=0.0):
    """La pérdida baja de forma más o menos recta en escala logarítmica, no exponencial.

    Con exp(-x/escala) sobre un eje log, toda la caída se amontona en el último tercio y durante
    media vuelta del bucle no se ve pasar nada.
    """
    y = suelo + alto * (1 - _T) ** codo + subida * _T ** 3
    return np.maximum(y * (1 + ruido * _rng.normal(size=len(XS))), 0.02)

ENTRENAMIENTO = _curva(0.85, 0.05, 0.025)
EXAMEN = _curva(0.72, 0.18, 0.030, subida=0.10)   # se queda arriba y repunta: memoriza
CAJA = (80, 740, rc.W - 80, 1300)
YLIM, YTICKS = (0.04, 1.3), [0.05, 0.1, 0.3, 1.0]
pct = lambda v: rc.dec(v * 100, 0) + "%"

def _upto(rf, n):
    """De la iteración 1 a XMAX a velocidad constante sobre el eje log."""
    return XMAX ** ((rf + 1) / n)

def s_curva(rf, n):
    img = rc.BGIMG.copy(); d = ImageDraw.Draw(img, "RGBA")
    rc.headline(d, ["ASÍ APRENDE", "UNA RED"], 420)
    u = _upto(rf, n)
    rc.chart(img, CAJA, [dict(xs=XS, ys=ENTRENAMIENTO, upto=u, col=rc.BLUE,
                              name="error", val_fmt=pct)],
             "error en entrenamiento", XMAX, YLIM, YTICKS, yfmt=pct)
    rc.chip(d, rc.W / 2, 1360, f"iteración {rc.fmt(int(u))}", anchor="mt")
    return img

def s_examen(rf, n):
    img = rc.BGIMG.copy(); d = ImageDraw.Draw(img, "RGBA")
    rc.headline(d, ["DOS CURVAS,", "UNA QUE IMPORTA"], 420)
    u = _upto(rf, n)
    rc.chart(img, CAJA, [dict(xs=XS, ys=ENTRENAMIENTO, upto=u, col=rc.BLUE, name="visto", val_fmt=pct),
                         dict(xs=XS, ys=EXAMEN, upto=u, col=rc.GOLD, name="examen", val_fmt=pct)],
             "error en entrenamiento y en examen", XMAX, YLIM, YTICKS, yfmt=pct)
    if u > XMAX * 0.45:
        rc.chip(d, rc.W / 2, 1360, "aquí empieza a memorizar", col=rc.GOLD, anchor="mt")
    return img

def s_cierre(rf, n):
    img = rc.BGIMG.copy(); d = ImageDraw.Draw(img, "RGBA")
    rc.headline(d, ["MIRA SIEMPRE", "LA DE ARRIBA"], 760)
    rc.body(d, ["la de abajo solo dice", "lo que la red ya se sabe"], 1000)
    a = rc.ease(rf / max(1, n - 1))
    rc.small(d, rc.HANDLE, 1400 + 20 * (1 - a), col=rc.MUTED)
    return img

FNS = {"curva": s_curva, "examen": s_examen, "cierre": s_cierre}
print("tres escenas listas")

## 4. La línea de tiempo

`resumen()` dice cuántas vueltas da cada dibujo. Si una escena da menos de dos, el ciclo es
demasiado largo para su frase y conviene acortarlo.

In [ ]:
TL = rv.TimelineNarrada(escenas, FNS)
print(TL.resumen())
if TL.N / rc.FPS > 23:
    print(f"\n[ojo] {TL.N / rc.FPS:.1f} s: por encima de los 18-23 s que pide el formato. "
          "Acorta una frase o quita una escena.")

# Un fotograma de cada escena, a media vuelta, para ver composición y zonas seguras.
for e in escenas:
    f = TL.start(e["id"]) + int(e["ciclo_s"] * rc.FPS / 2)
    display(f"── {e['id']} · fotograma {f}")
    display(TL.frame(f).resize((rc.W // 3, rc.H // 3), Image.LANCZOS))

## 5. Renderizar

Reanudable: los fotogramas que ya existen se saltan, así que puedes parar la celda y seguir
después. Medido en esta máquina: **0,05 s por fotograma**, unos 35 segundos para los 20 s de vídeo.

In [ ]:
NOMBRE = "demo_voz_curva"
carpeta = Path(f"work/frames_{NOMBRE}"); carpeta.mkdir(parents=True, exist_ok=True)
t0 = time.time(); hechos = 0
for f in range(TL.N):
    p = carpeta / f"{f:04d}.jpg"
    if p.exists():
        continue
    TL.frame(f).save(p, quality=95)
    hechos += 1
    if hechos % 60 == 0:
        print(f"  {f + 1}/{TL.N} · {(time.time() - t0) / hechos:.2f} s por fotograma", flush=True)
print(f"{hechos} fotogramas nuevos en {(time.time() - t0) / 60:.1f} min "
      f"· {len(list(carpeta.glob('*.jpg')))}/{TL.N} en disco")

In [ ]:
# Audio: la cama sintética del Reel, la voz encima y la música agachada mientras se habla.
A = rc.Audio(TL.N)
A.pad((98, 146.8, 196, 246.9))
for e in escenas[1:]:
    A.at(TL.start(e["id"]) - 6, A.whoosh(), 0.18)
rv.mezclar_voz(A, escenas, TL)

wav = f"work/{NOMBRE}.wav"; A.save(wav)
Path("out").mkdir(exist_ok=True)
subprocess.run(["ffmpeg", "-v", "error", "-y", "-framerate", str(rc.FPS),
                "-i", f"{carpeta}/%04d.jpg", "-i", wav, "-c:v", "libx264", "-pix_fmt", "yuv420p",
                "-crf", "18", "-c:a", "aac", "-b:a", "160k", "-shortest", f"out/{NOMBRE}.mp4"],
               check=True)
print(f"out/{NOMBRE}.mp4 · {TL.N / rc.FPS:.1f} s")

## 6. Juzgarlo

Cuatro preguntas, y la tercera es la que decide si esta idea del bucle vale:

1. **¿Se entiende sin sonido?** La mayoría lo verá en silencio la primera vez.
2. **¿La voz lenta (0,92) pega en formato corto?** Se calibró para episodios de doce minutos.
3. **¿El bucle ayuda o marea?** Ver la curva nacer tres veces puede fijar la idea o puede cansar.
   Si cansa, lo siguiente que hay que probar no es acortar el ciclo: es que cada vuelta añada algo
   (la primera pinta, la segunda marca el codo, la tercera se queda en la zona plana).
4. **¿El dorado compite con el azul de la curva?** En la serie el error es rojo; aquí el examen es
   dorado porque es la idea clave. Un espectador que venga de los episodios puede dudar.

In [ ]:
display(Video(f"out/{NOMBRE}.mp4", embed=True, width=360))
NOTAS = f"""
Duración:      {TL.N / rc.FPS:.1f} s ({len(escenas)} escenas)
Voz:           {voz['qwen']['modelo']} · velocidad {voz['velocidad']}

¿Se entiende sin sonido?   ...
¿La voz lenta encaja?      ...
¿El bucle ayuda o marea?   ...
¿Colores claros?           ...

Decisión: [ ] así se hacen los Reels  [ ] ajustar y repetir  [ ] volver al Timeline fijo
"""
Path("work/voz_demo/notas.md").write_text(NOTAS, encoding="utf-8")
print(NOTAS)